# Práctica P2: El Modelo de Overshooting de Dornbusch en Tiempo Discreto
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P2-v1.0
*   **Capítulo de Referencia**: Capítulo 3, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar el comportamiento dinámico de una pequeña economía abierta con movilidad perfecta de capitales. Estudiar cómo responde el tipo de cambio nominal a shocks monetarios y por qué experimenta una sobrerreacción o *overshooting* inicial debido a la rigidez temporal de los precios nacionales.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo de transmisión de la política monetaria en una economía abierta y cómo interactúan el mercado de bienes y el mercado de dinero.
2.  **Visualizar** el fenómeno de *overshooting* del tipo de cambio y su posterior convergencia dinámica.
3.  **Identificar** y analizar sistemas dinámicos con estabilidad de **punto de silla** (saddle point) en tiempo discreto.
4.  **Representar** e interpretar la transición de una economía en un **Diagrama de Fases** bidimensional que contenga curvas de demarcación, el camino de silla estable y el campo de vectores.


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

> *   **📋 Antes de empezar**, consulta ' (en esta misma carpeta): objetivos, tiempo estimado y conocimientos previos de esta práctica.
> 
> ⏱️ **~90-120 minutos**
> 
> 📋 **Prerrequisitos**: **Matemáticas**: sistemas de ecuaciones en diferencias, autovalores y punto de silla, diagramas de fases en tiempo discreto. | **Economía**: paridad descubierta de intereses (UIP), modelo IS-LM estático, neutralidad monetaria a largo plazo, relaci...
> 
### 🕹️ GUÍA RÁPIDA DE INICIO - Overshooting de Dornbusch
*   **¿Qué estamos haciendo aquí?** Analizamos cómo responde el tipo de cambio (el valor de la moneda) ante un shock.
*   **La gran idea (Sobrerreacción):** Como los precios de los supermercados tardan mucho en cambiar (son rígidos), el mercado financiero (tipo de cambio) reacciona con un "salto" exagerado al principio para compensar.
*   **¡Prueba esto!** Cambia la oferta monetaria `m0` en la simulación y observa la línea del tipo de cambio dar un salto enorme antes de estabilizarse lentamente en su nuevo valor.


In [ ]:
%%capture
# "%%capture" es una "magia" de Jupyter: ejecuta todo lo que hay debajo pero
# no imprime nada en pantalla. Útil para celdas de instalación que generan
# mucho texto de registro. La comprobación 'google.colab' in sys.modules
# mira si estamos en Colab: solo será True en ese entorno. En tu entorno
# local (venv), esta celda no hará nada.
import sys
if 'google.colab' in sys.modules:
    !pip install numpy scipy matplotlib ipywidgets

In [ ]:
# "import" trae a este cuaderno código escrito en otro sitio, para no tener
# que reescribirlo. "import X as np" trae toda la librería X y le da un
# alias corto (np). "from X import Y" es más selectivo: trae solo Y de X.
# sys.path.append('../../src') añade la carpeta src/ del proyecto al PATH
# para que Python encuentre macroaicomp aunque estemos dos niveles más abajo.

# Librerías de terceros (instaladas en el entorno con pip)
import numpy as np                      # cálculo numérico: vectores, matrices, álgebra lineal
import matplotlib.pyplot as plt         # visualización gráfica: figuras, ejes, estilos

# Controles interactivos (sliders) para modificar parámetros dinámicamente en Jupyter
from ipywidgets import interact, FloatSlider

# Proyecto (requiere `pip install -e .` desde la raíz del repo).
# La lógica del modelo Dornbusch (ecuaciones, simulación, estado estacionario,
# autovalores) vive en src/macroaicomp/models/dornbusch.py. El notebook solo
# llama funciones ya probadas, no reimplementa fórmulas.
import sys
sys.path.append('../../src')
from macroaicomp.models.dornbusch import (
    DornbuschParameters,
    default_calibration,
    steady_state,
    simulate_shock,
    eigenvalues,
    coefficient_matrices
)

## 1. El Marco Teórico: Ecuaciones y Estabilidad de Punto de Silla

El modelo de Dornbusch describe una economía pequeña y abierta con perfecta movilidad de capitales bajo las siguientes ecuaciones:

### 1.1 Ecuaciones Estructurales
1.  **Mercado Monetario (Curva LM):**
    $$m_t - p_t = \psi y^n_t - \theta i_t$$
    Donde $m$ es el logaritmo de la oferta monetaria, $p$ el log de precios, $y^n$ el log de producción potencial (asumimos $y_t = y^n_t$ para simplificar) e $i$ el tipo de interés nominal. Despejando el tipo de interés nominal:
    $$i_t = \frac{p_t - m_t + \psi y^n_t}{\theta}$$

2.  **Demanda Agregada en Economía Abierta (Curva IS):**
    $$y^d_t = \beta_0 + \beta_1(s_t - p_t + p^*_t) - \beta_2 i_t$$
    Donde $s$ es el log del tipo de cambio nominal (moneda nacional por moneda extranjera), $p^*$ los precios extranjeros e $y^d$ la demanda agregada. La demanda depende positivamente del tipo de cambio real (competitividad exterior) y negativamente del tipo de interés nominal.

3.  **Ajuste de Precios (Curva de Phillips en diferencias):**
    $$\Delta p_t = p_{t+1} - p_t = \mu(y^d_t - y^n_t)$$
    Donde $\mu$ representa la velocidad de ajuste del mercado de bienes.

4.  **Paridad No Cubierta de Intereses (UIP - Expectativas Racionales):**
    $$\Delta s_t = s_{t+1} - s_t = i_t - i^*_t$$
    Donde $i^*$ es el tipo de interés extranjero. Bajo previsión perfecta, la depreciación esperada del tipo de cambio coincide con la depreciación efectiva $\Delta s_t$.

---

### 1.2 Reducción a un Sistema Dinámico Lineal en Diferencias
Sustituyendo $i_t$ e $y^d_t$ en las ecuaciones de transición de $p_t$ y $s_t$, el sistema dinámico bidimensional se escribe en forma matricial como:
$$\begin{bmatrix} \Delta p_t \\ \Delta s_t \end{bmatrix} = \mathbf{A} \begin{bmatrix} p_t \\ s_t \end{bmatrix} + \mathbf{B} \mathbf{z}_t$$
Donde el vector de variables exógenas es $\mathbf{z}_t = [\beta_0, m_t, y^n_t, i^*_t, p^*_t]^T$ y las matrices son:
$$\mathbf{A} = \begin{bmatrix} -\mu\left( \beta_1 + \frac{\beta_2}{\theta} \right) & \mu\beta_1 \\ \frac{1}{\theta} & 0 \end{bmatrix}$$
$$\mathbf{B} = \begin{bmatrix} \mu & \frac{\mu\beta_{2}}{\theta} & - \mu\left(\frac{\psi\beta_{2}}{\theta} + 1\right) & 0 & \mu\beta_{1} \\ 0 & - \frac{1}{\theta} & \frac{\psi}{\theta} & - 1 & 0 \end{bmatrix}$$

Este sistema dinámico tiene una estructura de **punto de silla** (saddle point): posee un autovalor estable (cuyo módulo de $1+\lambda$ es menor a la unidad) y otro inestable. La variable de precios ($p$) es lenta y rígida a corto plazo, mientras que el tipo de cambio ($s$) es una variable forward-looking flexible que "salta" instantáneamente ante shocks para situar a la economía en la trayectoria de convergencia estable.


In [ ]:
# Esta celda solo FIJA NÚMEROS (Capítulo 3 del libro): todavía no calcula
# nada. default_calibration() devuelve un DornbuschParameters, un dataclass
# (definido en src/macroaicomp/models/dornbusch.py): una "ficha" con 10
# casillas con nombre (psi, theta, beta1, beta2, mi, beta0, m0, ypot0,
# pstar0, istar0). Usar default_calibration() evita escribir los números a
# mano y posibles errores de tecleo: los valores están centralizados en el
# código fuente y testeados. Al ejecutar veremos los 10 valores impresos con
# su descripción económica como comprobación visual.

# ==============================================================================
# CALIBRACIÓN DE PARÁMETROS BASE (Capítulo 3 - Libro original)
# ==============================================================================

# Cargar la calibración de referencia del modelo
params = default_calibration()

# Diccionario didáctico con la descripción económica de cada parámetro
descriptions = {
    "psi": "Sensibilidad de la demanda de dinero respecto al PIB [ψ]",
    "theta": "Sensibilidad de la demanda de dinero respecto al interés nominal [θ]",
    "beta1": "Sensibilidad de la demanda agregada respecto al tipo de cambio real [β1]",
    "beta2": "Sensibilidad de la demanda agregada respecto al interés nominal [β2]",
    "mi": "Velocidad de ajuste de precios ante excesos de demanda (Phillips) [μ]",
    "beta0": "Demanda agregada autónoma base [β0]",
    "m0": "Oferta monetaria nominal (logaritmo) [M0]",
    "ypot0": "Producción potencial (pleno empleo) [ypot]",
    "pstar0": "Logaritmo del nivel de precios extranjero [pstar]",
    "istar0": "Tipo de interés nominal extranjero (porcentaje) [istar]"
}

# Imprimir de forma estructurada los parámetros
print("CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):")
print("=" * 78)
print(f"{'Variable':<12} | {'Valor':<6} | {'Descripción Económica':<50}")
print("-" * 78)
for param_name, value in vars(params).items():
    desc = descriptions.get(param_name, "Parámetro del modelo")
    print(f"  {param_name:<10} | {value:<6} | {desc:<50}")
print("=" * 78)

## 2. Equilibrio de Largo Plazo (Estado Estacionario)

En el largo plazo, las variables se estabilizan, por lo que las variaciones temporales son nulas ($\Delta p_t = 0$ y $\Delta s_t = 0$). Resolviendo analíticamente:
1.  **De la ecuación de precios:** $\Delta p_t = 0 \Rightarrow y^d_t = y^n_t = \bar{Y}$.
2.  **De la ecuación del tipo de cambio:** $\Delta s_t = 0 \Rightarrow i_t = i^*_t$.
3.  **Del mercado de dinero:** Sustituyendo el interés en la demanda de saldos reales:
    $$p^* = m_t - \psi y^n_t + \theta i^*_t$$
4.  **Del mercado de bienes:** Sustituyendo las condiciones anteriores en la demanda agregada y despejando el tipo de cambio nominal de largo plazo:
    $$s^* = m_t - \frac{\beta_0}{\beta_1} + \left( \frac{1 - \psi\beta_1}{\beta_1} \right) y^n_t + \left( \frac{\theta\beta_1 + \beta_2}{\beta_1} \right) i^*_t - p^*_t$$

*Nota sobre fe errata del libro:* En el texto del Capítulo 3, por un error de imprenta, el denominador de la última fracción de la ecuación del tipo de cambio estacionario se imprime como $\beta_2$ en lugar de $\beta_1$. El código de la biblioteca `macroaicomp` implementa la fórmula matemáticamente correcta con el denominador $\beta_1$, lo que reproduce el valor numérico exacto de equilibrio del libro ($s^* = 76.52$).


In [ ]:
# steady_state() es una FUNCIÓN: le pasamos los parámetros y nos devuelve un
# diccionario con los valores de equilibrio de largo plazo (p, s, i, yd, dp,
# ds). Por dentro resuelve las condiciones Delta_p = 0, Delta_s = 0 usando
# las fórmulas analíticas derivadas en la Sección 2: p* = m - psi*ypot +
# theta*istar, s* = m - beta0/beta1 + ((1-psi*beta1)/beta1)*ypot +
# ((theta*beta1+beta2)/beta1)*istar - pstar.
# Las cadenas con "f" delante (f-strings) permiten meter valores dentro del
# texto. El formato .4f muestra 4 decimales fijos. El valor real guardado
# en la variable no cambia. Al ejecutar veremos: p*=1.5, s*=76.515,
# i*=3.0%, yd*=2000, dp=0, ds=0.

# ==============================================================================
# CÁLCULO NUMÉRICO DEL ESTADO ESTACIONARIO ANALÍTICO
# ==============================================================================

# Calcular el estado estacionario inicial a partir de las ecuaciones analíticas
ss = steady_state(params)

print("VALORES DE EQUILIBRIO DE LARGO PLAZO (ESTADO ESTACIONARIO BASE):")
print("-" * 65)
print(f"  Precios nacionales (p*)    : {ss['p']:.4f} (Log de precios)")
print(f"  Tipo de cambio nominal (s*): {ss['s']:.4f} (Log de moneda nacional/extranjera)")
print(f"  Tipo de interés (i*)       : {ss['i']:.2f}% (Igual al interés extranjero)")
print(f"  Demanda agregada (yd*)     : {ss['yd']:.2f} (Igual a producción potencial)")
print(f"  Tasa de inflación (dp)     : {ss['dp']:.4f} (Sin variaciones de precios)")
print(f"  Depreciación cambiaria (ds): {ss['ds']:.4f} (Sin variaciones del tipo de cambio)")
print("-" * 65)

In [ ]:
# Verificamos que el estado estacionario y los autovalores coinciden con el
# oráculo del Apéndice F (DYNARE) recogido en oraculo.md.
# np.testing.assert_allclose compara dos valores y SOLO lanza un error si la
# diferencia supera la tolerancia atol. No usamos "==" porque la aritmética
# con decimales casi nunca da resultados exactamente iguales (errores de
# redondeo internos). Si el port a Python tuviera un error, esta celda
# lanzaría AssertionError y detendría la ejecución antes de seguir
# construyendo gráficos sobre un resultado incorrecto.

# Verificar estado estacionario
np.testing.assert_allclose(ss['p'], 1.5, atol=1e-4)
np.testing.assert_allclose(ss['s'], 76.515, atol=1e-4)
np.testing.assert_allclose(ss['i'], 3.0, atol=1e-4)
np.testing.assert_allclose(ss['yd'], 2000.0, atol=1e-4)
np.testing.assert_allclose(ss['dp'], 0.0, atol=1e-4)
np.testing.assert_allclose(ss['ds'], 0.0, atol=1e-4)

# Verificar autovalores (punto de silla: uno estable, otro inestable)
lambdas = eigenvalues(params)
# Ordenamos para comparar: el estable (λ₁ ≈ −0.7415) y el inestable (λ₂ ≈ 0.5395)
lambdas_sorted = sorted(lambdas)
np.testing.assert_allclose(lambdas_sorted[0], -0.7415, atol=1e-4)
np.testing.assert_allclose(lambdas_sorted[1], 0.5395, atol=1e-4)
print("OK: coincide con el oráculo DYNARE (Apéndice F).")

## 3. Detrás de la Escena: El Salto del Tipo de Cambio al Camino de Silla Estable

En un sistema dinámico con un punto de silla, si la economía sufre un shock imprevisto, la única forma de evitar que las variables exploten y diverjan hacia el infinito a largo plazo es que la variable flexible (el tipo de cambio, $s$) **salte instantáneamente en el periodo del shock** hacia la trayectoria estable (el *camino de silla* o *stable path*).

### La Ecuación del Salto de Expectativas
La trayectoria de convergencia estable del tipo de cambio nominal está gobernada por el autovalor estable $\lambda_1$ (el que cumple $|1+\lambda_1| < 1$):
$$\Delta s_t = \lambda_1(s_t - \bar{s}_t)$$
Igualando esta condición de convergencia con la ecuación de Paridad No Cubierta de Intereses (UIP) de la economía en el momento del shock ($t=1$), podemos despejar el valor exacto al que debe saltar el tipo de cambio:
$$\lambda_1(s_1 - \bar{s}_1) = -\frac{1}{\theta}(m_1 - p_1 - \psi y^n_1) - i^*_1$$
$$s_1 = \frac{-(m_1 - p_1 - \psi y^n_1)}{\theta \lambda_1} - \frac{i^*_1}{\lambda_1} + \bar{s}_1$$

Dado que los precios son rígidos en el primer período, $p_1$ se mantiene en su nivel antiguo pre-shock. Sin embargo, dado que $\lambda_1$ es negativo (aproximadamente $-0.74$), el tipo de cambio nominal experimenta una **sobrerreacción inicial** ($s_1$ sube de golpe por encima de su nivel de equilibrio final $\bar{s}_1$).


In [ ]:
# Esta celda DEFINE (no ejecuta) simulate_dornbusch_manual(), una versión
# didáctica de simulate_shock() que desglosa paso a paso la recursión en
# diferencias del modelo de Dornbusch. El objetivo es que veas CÓMO funciona
# el salto del tipo de cambio (overshooting): en el periodo del shock, el
# tipo de cambio s NO sigue la ecuación estándar, sino que salta
# directamente al "camino de silla estable" (stable path) usando el
# autovalor estable lambda1. Los precios p, en cambio, son rígidos: no se
# mueven en el periodo del shock. Esta es la esencia del modelo: una
# variable "de salto" (s, forward-looking) y otra "predeterminada" (p, lenta).
# "def nombre(args):" define una FUNCIÓN: no se ejecuta al escribirla, solo
# cuando alguien la llama (la celda siguiente la llama para verificarla).

# ==============================================================================
# SIMULACIÓN MANUAL DE LA DINÁMICA CON JUMP DE EXPECTATIVAS
# ==============================================================================

def simulate_dornbusch_manual(params, z_init, z_final, periods=30, shock_period=1):
    """
    Función explicativa que simula paso a paso la dinámica de Dornbusch en diferencias.
    """
    # 1. Obtener autovalores de la matriz del sistema para identificar el autovalor estable
    lambdas = eigenvalues(params)
    stable_idx = np.argmin(np.abs(lambdas + 1.0))
    stable_lambda = lambdas[stable_idx]  # lambda1 (approx -0.74)
    
    # 2. Inicializar arrays de trayectorias
    p = np.zeros(periods)
    s = np.zeros(periods)
    
    # Calcular estados estacionarios
    # Pasamos las variables exógenas z = [beta0, m, ypot, istar, pstar]
    p_ss_init = z_init[1] - params.psi * z_init[2] + params.theta * z_init[3]
    s_ss_final = (
        z_final[1]
        - z_final[0] / params.beta1
        + ((1.0 - params.psi * params.beta1) / params.beta1) * z_final[2]
        + ((params.theta * params.beta1 + params.beta2) / params.beta1) * z_final[3]
        - z_final[4]
    )
    
    # Periodo t = 0: Estado estacionario inicial
    p[0] = p_ss_init
    s[0] = z_init[1] - z_init[0]/params.beta1 + ((1.0 - params.psi*params.beta1)/params.beta1)*z_init[2] + ((params.theta*params.beta1 + params.beta2)/params.beta1)*z_init[3] - z_init[4]
    
    # Evolución temporal paso a paso
    for t in range(1, periods):
        if t == shock_period:
            # Precios rígidos a corto plazo
            p[t] = p[t - 1]
            # Salto del tipo de cambio al stable path (Overshooting)
            m_1 = z_final[1]
            ypot_1 = z_final[2]
            istar_1 = z_final[3]
            s[t] = (
                -(m_1 - p[t] - params.psi * ypot_1) / (params.theta * stable_lambda)
                - istar_1 / stable_lambda
                + s_ss_final
            )
        else:
            # Propagación estándar para periodos posteriores usando las matrices A y B
            # p_t = p_{t-1} + dp_{t-1}
            # s_t = s_{t-1} + ds_{t-1}
            z_curr = z_final if t >= shock_period else z_init
            i_prev = -(z_curr[1] - p[t-1] - params.psi * z_curr[2]) / params.theta
            yd_prev = z_curr[0] + params.beta1 * (s[t-1] - p[t-1] + z_curr[4]) - params.beta2 * i_prev
            dp_prev = params.mi * (yd_prev - z_curr[2])
            ds_prev = i_prev - z_curr[3]
            
            p[t] = p[t-1] + dp_prev
            s[t] = s[t-1] + ds_prev
            
    return p, s

print("Función de simulación manual registrada con éxito.")

## 4. Transmisión Económica e Interactividad (Diagrama de Fases)

### 4.1 El Mecanismo de Overshooting
Cuando la oferta monetaria se incrementa ($M_0 \uparrow$):
1.  **A corto plazo:** Los precios nacionales son rígidos ($P_1 = P_0$). La liquidez agregada reduce el interés nacional ($i_1 \downarrow$). Como el tipo extranjero $i^*$ es constante, la UIP exige una expectativa de apreciación cambiaria futura. Para que el tipo de cambio pueda apreciarse en el futuro y a la vez converger a un nivel de largo plazo que es más depreciado, el tipo de cambio nominal **debe experimentar un salto vertical inicial sobredimensionado** ($s_1 \uparrow \uparrow$).
2.  **Durante la transición:** El tipo de cambio real depreciado y los tipos bajos expanden la demanda agregada ($Y^d_1 > Y^n$). Esto genera inflación gradual ($\Delta p_t > 0$). Conforme $P$ sube, la liquidez se contrae, $i$ sube de vuelta a $i^*$ y el tipo de cambio nominal se aprecia progresivamente deslizándose por el camino de silla estable.

### 4.2 Locus y Camino de Silla en el Gráfico de Fases
*   **Locus $\Delta s_t = 0$ (Equilibrio de Dinero):** Línea vertical en el nivel de precios estacionario $p_t = p^*$.
*   **Locus $\Delta p_t = 0$ (Equilibrio de Bienes):** Línea con pendiente positiva ($1.01$ en calibración base) dada por:
    $$s_t = \frac{\beta_0 + p^*_t \beta_1 + \frac{\beta_2}{\theta}(m_t - \psi y^n_t) - y^n_t \left(1 + \frac{\psi \beta_2}{\theta}\right)}{\beta_1} + \left(1 + \frac{\beta_2}{\theta \beta_1}\right) p_t$$
*   **Saddle Path (Camino de Silla Estable):** Línea con pendiente negativa ($k \approx -2.70$):
    $$s_t - \bar{s} = k(p_t - \bar{p})$$


In [ ]:
# "def nombre(args):" define una FUNCIÓN reutilizable: no se ejecuta al
# escribirla, solo cuando interact() la llama más abajo.
# plot_dornbusch_response() simula el modelo de Dornbusch con los valores de
# m0 y beta0 que elijas en los sliders y dibuja 3 paneles: (1) trayectorias
# temporales de s y p, (2) tipo de interés y demanda agregada, (3) diagrama
# de fases en el plano (p, s) con locus Delta_s=0, Delta_p=0, el camino de
# silla estable (saddle path) y el campo vectorial. simulate_shock() aplica
# la recursión en diferencias con salto del tipo de cambio en t=1.
# Al mover los sliders verás en vivo cómo cambia el overshooting.
# interact() conecta la función a los FloatSlider: cada vez que mueves uno,
# ipywidgets vuelve a llamar a plot_dornbusch_response() con los nuevos
# valores y redibuja los 3 paneles. El ";" al final evita que Jupyter
# imprima también la representación de texto del widget.

# ==============================================================================
# SIMULACIÓN INTERACTIVA DE SHOCKS Y REPRESENTACIÓN EN 3 PANELES (CON DIAGRAMA DE FASES)
# ==============================================================================

def plot_dornbusch_response(m0_shock=101.0, beta0_shock=500.0):
    """
    Simula la respuesta del modelo de Dornbusch ante shocks en la oferta de dinero
    y el gasto autónomo, mostrando trayectorias temporales y el diagrama de fases.
    """
    # 1. Definir los vectores exógenos iniciales y finales
    # z = [beta0, m, ypot, istar, pstar]
    z_initial = np.array([500.0, 100.0, 2000.0, 3.0, 0.0])
    z_final = np.array([beta0_shock, m0_shock, 2000.0, 3.0, 0.0])
    
    # 2. Cargar calibración base
    params_sim = default_calibration()
    ss_init = steady_state(params_sim)
    
    # 3. Simular trayectoria usando el módulo de librería
    periods = 30
    res = simulate_shock(params_sim, z_initial, z_final, periods=periods, shock_period=1)
    ss_final = steady_state(
        DornbuschParameters(
            beta0=z_final[0], m0=z_final[1], ypot0=z_final[2], istar0=z_final[3], pstar0=z_final[4]
        )
    )
    
    # Obtener autovalores y el estable para dibujar los locus
    lambdas = eigenvalues(params_sim)
    stable_idx = np.argmin(np.abs(lambdas + 1.0))
    stable_lambda = lambdas[stable_idx]
    
    # 4. Crear la figura de 3 paneles
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    t_axis = np.arange(periods)
    
    # --- PANEL 1: DINÁMICA TEMPORAL DE PRECIOS Y TIPO DE CAMBIO ---
    axs[0].plot(t_axis, res["s"], color='#7A3E9F', linewidth=2.5, label='Tipo de cambio (s)')
    axs[0].plot(t_axis, res["p"], color='#8EAD3A', linewidth=2.5, label='Precios (p)')
    axs[0].axhline(ss_init["s"], color='#7A3E9F', linestyle=':', alpha=0.5, label='s Inicial')
    axs[0].axhline(ss_final["s"], color='#7A3E9F', linestyle='--', alpha=0.7, label='s Final')
    axs[0].axhline(ss_init["p"], color='#8EAD3A', linestyle=':', alpha=0.5, label='p Inicial')
    axs[0].axhline(ss_final["p"], color='#8EAD3A', linestyle='--', alpha=0.7, label='p Final')
    axs[0].axvline(1, color='red', linestyle='--', alpha=0.4, label='Momento del Shock (t=1)')
    axs[0].set_title('Trayectorias Temporales (s y p)', fontsize=11, fontweight='bold', pad=10)
    axs[0].set_xlabel('Tiempo (t)', fontsize=9)
    axs[0].set_ylabel('Escala Logarítmica', fontsize=9)
    axs[0].grid(True, linestyle=':', alpha=0.6)
    axs[0].legend(loc='best', fontsize=8)
    
    # --- PANEL 2: INTERÉS Y DEMANDA AGREGADA ---
    axs[1].plot(t_axis, res["i"], color='#004C97', linewidth=2.0, label='Tipo interés (i)')
    axs[1].plot(t_axis, res["yd"], color='#D95319', linewidth=2.0, label='Demanda agregada (yd)')
    axs[1].axhline(params_sim.istar0, color='#004C97', linestyle='--', alpha=0.5, label='Interés extranjero (i*)')
    axs[1].axhline(params_sim.ypot0, color='#D95319', linestyle='--', alpha=0.5, label='PIB Potencial (ypot)')
    axs[1].axvline(1, color='red', linestyle='--', alpha=0.4)
    axs[1].set_title('Tipos de Interés y Demanda Agregada', fontsize=11, fontweight='bold', pad=10)
    axs[1].set_xlabel('Tiempo (t)', fontsize=9)
    axs[1].grid(True, linestyle=':', alpha=0.6)
    axs[1].legend(loc='best', fontsize=8)
    
    # --- PANEL 3: DIAGRAMA DE FASES EN EL PLANO (p, s) ---
    # Dibujar la trayectoria simulada
    axs[2].plot(res["p"], res["s"], color='#7A3E9F', linewidth=3, label='Trayectoria dinámica')
    
    # Dibujar el locus ds_t = 0 (Línea vertical en p_ss_final)
    axs[2].axvline(ss_final["p"], color='#004C97', linestyle='--', linewidth=2, label=r'$\Delta s = 0$ (Locus final)')
    axs[2].axvline(ss_init["p"], color='#004C97', linestyle=':', alpha=0.5, label=r'$\Delta s = 0$ (Locus inicial)')
    
    # Dibujar el locus dp_t = 0 (Línea inclinada ascendente)
    p_vals = np.linspace(min(res["p"]) - 0.5, max(res["p"]) + 0.5, 100)
    
    # Ecuación del locus dp_t = 0: s = C_locus + (1 + beta2/(theta*beta1))*p
    slope_dp = 1.0 + params_sim.beta2 / (params_sim.theta * params_sim.beta1)
    c_locus_init = ss_init["s"] - slope_dp * ss_init["p"]
    c_locus_final = ss_final["s"] - slope_dp * ss_final["p"]
    
    s_locus_init = c_locus_init + slope_dp * p_vals
    s_locus_final = c_locus_final + slope_dp * p_vals
    
    axs[2].plot(p_vals, s_locus_init, color='#8EAD3A', linestyle=':', alpha=0.5, label=r'$\Delta p = 0$ (Inicial)')
    axs[2].plot(p_vals, s_locus_final, color='#8EAD3A', linestyle='--', linewidth=2, label=r'$\Delta p = 0$ (Final)')
    
    # Dibujar el stable saddle path (pendiente negativa) que pasa por el nuevo steady state
    a_mat, _ = coefficient_matrices(params_sim)
    k_slope = (stable_lambda - a_mat[0,0]) / a_mat[0,1]
    saddle_final = ss_final["s"] + k_slope * (p_vals - ss_final["p"])
    axs[2].plot(p_vals, saddle_final, color='black', linestyle='-.', alpha=0.8, label='Camino de Silla Estable')
    
    # Dibujar el campo vectorial (quiver) en el plano (p, s)
    p_grid, s_grid = np.meshgrid(
        np.linspace(min(res["p"]) - 0.3, max(res["p"]) + 0.3, 10),
        np.linspace(min(res["s"]) - 0.5, max(res["s"]) + 0.5, 10)
    )
    dp_grid = np.zeros_like(p_grid)
    ds_grid = np.zeros_like(s_grid)
    for r in range(p_grid.shape[0]):
        for c in range(p_grid.shape[1]):
            # Calcular derivadas del sistema lineal en diferencias en cada cuadrante
            i_pt = -(z_final[1] - p_grid[r, c] - params_sim.psi * z_final[2]) / params_sim.theta
            yd_pt = z_final[0] + params_sim.beta1 * (s_grid[r, c] - p_grid[r, c] + z_final[4]) - params_sim.beta2 * i_pt
            dp_grid[r, c] = params_sim.mi * (yd_pt - z_final[2])
            ds_grid[r, c] = i_pt - z_final[3]
            
    norm = np.hypot(dp_grid, ds_grid)
    norm[norm == 0] = 1.0
    dp_grid /= norm
    ds_grid /= norm
    axs[2].quiver(p_grid, s_grid, dp_grid, ds_grid, color='lightgray', alpha=0.6, scale=25, width=0.003)
    
    # Puntos singulares
    axs[2].scatter(ss_init["p"], ss_init["s"], color='gray', s=100, zorder=5, label='E.E. Inicial')
    axs[2].scatter(ss_final["p"], ss_final["s"], color='black', marker='*', s=200, zorder=5, label='E.E. Final')
    
    # Mostrar el vector del salto instantáneo (de G3 a G4 en p=1.5)
    axs[2].annotate('', xy=(res["p"][1], res["s"][1]), xytext=(res["p"][0], res["s"][0]),
                     arrowprops=dict(facecolor='#7A3E9F', shrink=0.05, width=1.5, headwidth=8))
    axs[2].text(res["p"][0] - 0.15, (res["s"][0] + res["s"][1])/2, "Jump", color='#7A3E9F', fontweight='bold')
    
    # Ajustes estéticos
    axs[2].set_title('Plano de Fases (p, s)', fontsize=11, fontweight='bold', pad=10)
    axs[2].set_xlabel('Precios (p)', fontsize=9)
    axs[2].set_ylabel('Tipo de cambio (s)', fontsize=9)
    axs[2].set_xlim(min(res["p"]) - 0.4, max(res["p"]) + 0.4)
    axs[2].set_ylim(min(res["s"]) - 0.8, max(res["s"]) + 0.8)
    axs[2].grid(True, linestyle=':', alpha=0.6)
    axs[2].legend(loc='best', fontsize=7)
    
    plt.tight_layout()
    plt.show()

# Configurar sliders interactivos
interact(
    plot_dornbusch_response,
    m0_shock=FloatSlider(value=101.0, min=98.0, max=104.0, step=0.5, description='Dinero (M)'),
    beta0_shock=FloatSlider(value=500.0, min=450.0, max=550.0, step=10.0, description='Gasto (B0)')
);

## 5. Verificación frente al oráculo

Comparamos contra los valores reportados en el libro y reproducidos por el
código DYNARE del Apéndice F (`referencia/icm3.mod`), recogidos en
`oraculo.md`:

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Estado estacionario inicial — $p$ | 1.5 |
| Estado estacionario inicial — $s$ | 76.515 |
| Estado estacionario inicial — $i$ | 3.0 |
| Estado estacionario inicial — $y^d$ | 2000.0 |
| Estado estacionario inicial — $\Delta p$ | 0.0 |
| Estado estacionario inicial — $\Delta s$ | 0.0 |
| Autovalor estable $\lambda_1$ | −0.7415 |
| Autovalor inestable $\lambda_2$ | 0.5395 |
| $p$ en $t=1$ (shock $M_0: 100 \to 101$) | 1.5 (sticky) |
| $s$ en $t=1$ (overshooting) | 80.215 |
| $i$ en $t=1$ (cae por shock expansivo) | 1.0 |
| $p$ en largo plazo ($t=29$, nuevo SS) | 2.5 |
| $s$ en largo plazo ($t=29$, nuevo SS) | 77.515 |
| $i$ en largo plazo (vuelve a $i^*$) | 3.0 |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.

In [ ]:
# Verificamos los valores del shock monetario (M0: 100 -> 101) contra el
# oráculo del Apéndice F (DYNARE) recogido en oraculo.md.
# La tolerancia es más laxa en el salto de s (5e-3) y en el largo plazo
# (1e-2) porque la simulación acumula errores de redondeo en 30 periodos.

# Configurar shock monetario: m0 de 100 a 101
import numpy as np
from macroaicomp.models.dornbusch import DornbuschParameters
z_initial = np.array([500.0, 100.0, 2000.0, 3.0, 0.0])
z_final = np.array([500.0, 101.0, 2000.0, 3.0, 0.0])
params_sim = default_calibration()
res_shock = simulate_shock(params_sim, z_initial, z_final, periods=30, shock_period=1)

# Verificar valores en t=1 (periodo del shock)
np.testing.assert_allclose(res_shock['p'][1], 1.5, atol=1e-4)   # precios rígidos
np.testing.assert_allclose(res_shock['s'][1], 80.215, atol=5e-3) # overshooting
np.testing.assert_allclose(res_shock['i'][1], 1.0, atol=1e-3)    # cae por shock expansivo

# Verificar valores en largo plazo (t=29)
np.testing.assert_allclose(res_shock['p'][29], 2.5, atol=1e-2)
np.testing.assert_allclose(res_shock['s'][29], 77.515, atol=1e-2)
np.testing.assert_allclose(res_shock['i'][29], 3.0, atol=1e-2)   # vuelve a i*
print("OK: shock monetario coincide con el oráculo DYNARE (Apéndice F).")

## 6. Buenas Prácticas Aplicadas en este Laboratorio

Fíjate en las siguientes decisiones de diseño técnico que hacen de este código un estándar ejemplar:
1.  **Higiene de Datos de Entrada**: Se aíslan los parámetros estructurales en un dataclass `DornbuschParameters` para evitar valores ocultos dentro de las ecuaciones.
2.  **Lógica Externa Reutilizable**: Las rutinas matriciales complejas residen en `src/macroaicomp/models/dornbusch.py` facilitando el testeo y mantenimiento.
3.  **Higiene de Control de Versiones**: Las celdas de este cuaderno se han limpiado de outputs volátiles mediante `nbstripout` antes de confirmar la versión en el repositorio de código.


## Extensiones para ABP (Aprendizaje Basado en Proyectos)

1. **Shock fiscal**: simular un aumento del gasto público ($\beta_0$) y analizar si produce overshooting o undershooting, y por qué.
2. **Ajuste gradual de expectativas**: modificar la UIP para que las expectativas cambiarias sean adaptativas en vez de racionales (ej. $\Delta s^e = \lambda(s^* - s)$) y comparar la dinámica.
3. **Calibración con datos reales**: calibrar $\mu$ y $\nu$ con datos trimestrales de inflación y brecha de producción de una economía real (ej. España 1999-2019) y simular un shock de política monetaria.

## 7. Cuaderno de Bitácora (Actividades para el Alumno)

Responde en tu Cuaderno de Bitácora las siguientes preguntas analíticas tras experimentar con la simulación interactiva:

1.  **El Experimento del Overshooting Monetario:** Restaura el Gasto Autónomo ($\beta_0$) a $500$ e incrementa la Oferta Monetaria ($M_0$) de $100$ a $101$ en el deslizador.
    *   ¿Por qué el tipo de cambio ($s$) salta de golpe a $80.22$, que es un nivel significativamente superior al nuevo equilibrio de largo plazo ($77.52$)? Explica el rol de la rigidez de precios en este comportamiento de sobrerreacción.
    *   Describe qué ocurre con el tipo de interés nominal doméstico ($i$) en el periodo 1 y cómo influye en la depreciación instantánea del tipo de cambio según la UIP.
2.  **Sensibilidad respecto a la Rigidez de Precios ($\mu$):**
    *   Modifica conceptualmente la velocidad de ajuste de precios $\mu$. Si los precios fuesen **aún más rígidos** (por ejemplo, $\mu=0.001$), ¿el salto (overshooting) del tipo de cambio sería mayor o menor que con la calibración base ($\mu=0.01$)? Justifica tu hipótesis a partir de la fórmula del salto.
3.  **Análisis del Diagrama de Fases (Plano de Estados):**
    *   Explica visualmente a partir del Panel 3 por qué la trayectoria de transición no es una línea recta directa desde el equilibrio inicial al equilibrio final.
    *   ¿Qué representa el segmento vertical denotado como "Jump" y por qué se produce de forma instantánea sin variaciones en el eje horizontal de precios?
    *   Describe qué ocurre una vez que el sistema alcanza la línea del Camino de Silla Estable. ¿Hacia dónde se desliza y cuál es la dirección de las fuerzas del campo vectorial?
